# Assignment 6: Object Detection using Transfer Learning of CNN Architectures

**Aim:** Implementation of object detection using transfer learning of CNN architectures.

**Model used:** Pre-trained YOLOv8n.

**Important before running:**
- Keep this notebook in a folder where you can save the training outputs.
- The first run needs internet to install Ultralytics and download `yolov8n.pt`, `coco8.yaml`, and the COCO8 dataset (unless already available).
- Training is performed in three phases: frozen backbone, partial fine-tuning and full fine-tuning.
- This Jupyter version uses Matplotlib to display the output image instead of Google Colab's `cv2_imshow`.


## Cell 1: Install required package
Run this cell once. If Ultralytics is already installed, you can skip it.

In [ ]:
%pip install ultralytics

## Cell 2: Import libraries

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Libraries imported successfully.")

## Cell 3: Load the pre-trained YOLOv8 model and count parameters

In [ ]:
# Load a pre-trained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Count total and trainable parameters
total = sum(p.numel() for p in model.model.parameters())

trainable = sum(
    p.numel() for p in model.model.parameters()
    if p.requires_grad
)

print("Total Parameters:", total)
print("Trainable Parameters:", trainable)

## Cell 4: Set common training parameters

In [ ]:
data = "coco8.yaml"
imgsz = 640
batch = 16

print("Dataset:", data)
print("Image size:", imgsz)
print("Batch size:", batch)

## Cell 5: Phase 1 – Freeze initial layers and train detection head

The first 10 layers are frozen. The remaining trainable layers are trained.

In [ ]:
print("\n=== Phase 1: Backbone Frozen ===")

model.train(
    data=data,
    epochs=10,
    imgsz=imgsz,
    batch=batch,
    freeze=10,
    lr0=1e-3,
    name="phase1_head",
    plots=False,
    workers=2,
    optimizer="AdamW"
)

best1 = str(model.trainer.best)
print("Best weights from Phase 1:", best1)

## Cell 6: Phase 2 – Partial fine-tuning

Load the best weights from Phase 1. Freeze fewer layers so that more of the network can be trained.

In [ ]:
print("\n=== Phase 2: Partial Fine-Tuning ===")

model = YOLO(best1)

model.train(
    data=data,
    epochs=10,
    imgsz=imgsz,
    batch=batch,
    freeze=5,
    lr0=3e-4,
    name="phase2_partial",
    plots=False,
    workers=2,
    optimizer="AdamW"
)

best2 = str(model.trainer.best)
print("Best weights from Phase 2:", best2)

## Cell 7: Phase 3 – Unfreeze all layers and perform full fine-tuning

Load the best weights from Phase 2 and train all layers.

In [ ]:
print("\n=== Phase 3: Full Fine-Tuning ===")

model = YOLO(best2)

model.train(
    data=data,
    epochs=10,
    imgsz=imgsz,
    batch=batch,
    freeze=None,
    lr0=1e-4,
    name="phase3_full",
    plots=False,
    workers=2,
    optimizer="AdamW"
)

best3 = str(model.trainer.best)
print("Best weights from Phase 3:", best3)

## Cell 8: Evaluate the final model

In [ ]:
final = YOLO(best3)

metrics = final.val(
    data=data,
    imgsz=imgsz
)

print("\nFinal mAP50:", metrics.box.map50)
print("Final mAP50-95:", metrics.box.map)
print("Best weights:", best3)

## Cell 9: Object detection and bounding-box visualization

**Image input:** Put an image named `bus.jpg` in the same folder as this notebook. Alternatively, change `image_path` below to the full path of an image on your computer.

In [ ]:
def detect_and_draw(model, image_path, conf=0.25):
    # Run object detection
    result = model.predict(
        source=image_path,
        conf=conf,
        verbose=False
    )[0]

    # Original image in OpenCV BGR format
    img = result.orig_img.copy()

    rng = np.random.default_rng(42)
    colors = {
        i: tuple(int(c) for c in rng.integers(50, 255, 3))
        for i in result.names
    }

    for box in result.boxes:
        cls_id = int(box.cls)
        label = f"{result.names[cls_id]} {float(box.conf):.2f}"

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        color = colors[cls_id]

        # Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

        # Calculate label size
        (tw, th), base = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2
        )

        top = max(y1 - th - base, 0)

        # Draw label background and text
        cv2.rectangle(
            img, (x1, top), (x1 + tw, top + th + base), color, -1
        )
        cv2.putText(
            img, label, (x1, top + th),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
            (255, 255, 255), 2, cv2.LINE_AA
        )

    # Save output image
    cv2.imwrite("prediction.jpg", img)

    # Display image in Jupyter Notebook
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Object Detection using Transfer Learning")
    plt.axis("off")
    plt.show()

    return img


# Change this path if your image has a different name or location
image_path = "bus.jpg"

if os.path.exists(image_path):
    result = detect_and_draw(final, image_path)
    print("Output saved as prediction.jpg")
else:
    print("Image not found:", image_path)
    print("Place bus.jpg in the same folder as this notebook, or update image_path.")

## Conclusion

Thus, we implemented object detection using transfer learning with a pre-trained YOLOv8 model. We froze the initial layers, performed partial fine-tuning, and then fine-tuned all layers. Finally, we evaluated the model and visualized detected objects with bounding boxes.

**Note:** This implementation fine-tunes the detection head already present in YOLOv8; it does not manually add a separate custom classifier.